# Thermodynamic ETH diagnostics for square-QDM cage witnesses

This notebook demonstrates the thermodynamic-limit workflow in `qlinks`:

1. define or extract a size-independent local row operator $L_R$;
2. evaluate $Q_R=L_R^\dagger L_R$ in the exact constrained basis;
3. reproduce the same infinite-temperature expectation with a strip transfer matrix;
4. project the transfer contraction into the same electric winding sector;
5. compare charge-resolved dynamic programming with Fourier projection;
6. increase the strip length or circumference without enumerating the global dimer basis.


In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qlinks.caging import (
    CageClassificationConfig,
    LocalQDMCageSearchConfig,
    RobustQDMLocalCageSearchConfig,
    SquareQDMBiperiodicProductTile,
    SquareQDMBiperiodicTileSearchConfig,
    SquareQDMPeriodicProductUnitCell,
    LocalWitnessEmbeddingRecord,
    LocalWitnessFamily,
    ReducedIZPatternSupport,
    SquareQDMStripTransferMatrix,
    SquareQDMStripWindingSector,
    SquareQDMWitnessPlacement,
    certify_local_witness_on_square_qdm_periodic_sequence,
    certify_square_qdm_periodic_product_sequence,
    diagnose_square_qdm_biperiodic_repeatability,
    classify_cage_state,
    evaluate_local_witness_on_diagonal_ensemble,
    evaluate_square_qdm_classification_witnesses_on_strips,
    evaluate_square_qdm_witness_family_on_strips,
    local_witness_template_from_pattern_support,
    local_witnesses_from_classification_report,
    robust_qdm_local_cage_search,
    scan_square_qdm_beta_zero_energy_density,
    search_square_qdm_biperiodic_product_tiles,
)
from qlinks.models import SquareQDMModel

## 1. Build a local witness

For a cage calculation, the usual entry point is
`local_witnesses_from_classification_report(report)`.  To keep this notebook self-contained, we use one directed plaquette flip,

$$
L_R = |0101\rangle\langle 1010|,
$$

on the four ordered links of one square plaquette.  Then $Q_R$ is the projector onto the source flippable orientation after projection into the constrained dimer Hilbert space.


In [ ]:
reference_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
)

plaquette_id = reference_model.lattice.plaquette_id_from_cell(0, 0)
variable_indices = tuple(
    reference_model.layout.link_variable_index(int(link_id))
    for link_id in reference_model.lattice.plaquette_links(plaquette_id)
)

pattern_support = ReducedIZPatternSupport(
    pattern_key=(
        (
            (1, 0, 1, 0),
            (0, 1, 0, 1),
            (1.0, 0.0),
        ),
    ),
    variable_indices=variable_indices,
    source_zero_indices=(),
    mechanism_labels=(),
)

template = local_witness_template_from_pattern_support(pattern_support)
witness = template.instantiate(variable_indices)
template.to_summary_dict()

## 2. Exact constrained-basis value on the $4\times4$ torus

This is the reference calculation.  It constructs all dimer coverings but does not require diagonalizing the Hamiltonian because the infinite-temperature state is diagonal and uniform in the constrained basis.


In [ ]:
reference_basis = reference_model.build_basis(solver="dfs")
exact_4x4 = evaluate_local_witness_on_diagonal_ensemble(
    witness,
    basis_configs=reference_basis.states,
)

pd.DataFrame(
    [
        {
            "basis dimension": reference_basis.n_states,
            "<Q_R>_infinite_temperature": exact_4x4.expectation,
            "Var(Q_R)": exact_4x4.variance,
        }
    ]
)

## 3. Convert the finite embedding to strip coordinates

`SquareQDMWitnessPlacement.from_local_witness` removes global variable labels and records the physical square-lattice link coordinates.  If the witness crosses the periodic x seam, the shortest unwrapped interval is chosen automatically.


In [ ]:
placement = SquareQDMWitnessPlacement.from_local_witness(
    reference_model,
    witness,
)
transfer = SquareQDMStripTransferMatrix(circumference=reference_model.ly)

{
    "link_coordinates": placement.link_coordinates,
    "window_width": placement.window_width,
    "affected_sites": placement.affected_sites,
    "reference_origin_x": placement.reference_origin_x,
    "boundary_state_count": transfer.n_boundary_states,
}

## 4. Exact transfer-matrix check on the same torus

For periodic x boundaries, the transfer calculation counts the same $4\times4$ torus coverings.  The weighted count is the exact trace of $Q_R$ over that constrained Hilbert space.


In [ ]:
transfer_4x4 = transfer.evaluate_witness(
    placement,
    length=reference_model.lx,
    boundary_x="periodic",
)

comparison = pd.DataFrame(
    [
        {
            "method": "explicit constrained basis",
            "partition count": reference_basis.n_states,
            "weighted count": exact_4x4.expectation * reference_basis.n_states,
            "<Q_R>": exact_4x4.expectation,
        },
        {
            "method": "strip transfer matrix",
            "partition count": transfer_4x4.partition_count,
            "weighted count": transfer_4x4.weighted_count,
            "<Q_R>": transfer_4x4.expectation,
        },
    ]
)

assert np.isclose(transfer_4x4.expectation, exact_4x4.expectation)
comparison

## 5. Resolve the periodic torus by electric winding sector

The transfer boundary mask fixes the electric $x$ winding, while each column transition carries an additive electric $y$-winding charge.  This gives the exact same-sector thermal comparison required by ETH.


In [ ]:
sector_counts = transfer.periodic_winding_sector_counts(length=reference_model.lx)
sector_table = pd.DataFrame(
    [
        {"winding_x": sector.winding_x, "winding_y": sector.winding_y, "dimension": count}
        for sector, count in sorted(sector_counts.items(), key=lambda item: item[0].label)
    ]
)
assert np.isclose(sector_table["dimension"].sum(), reference_basis.n_states)
sector_table

In [ ]:
w00 = SquareQDMStripWindingSector(winding_x=0, winding_y=0)
transfer_w00 = transfer.evaluate_witness(
    placement,
    length=reference_model.lx,
    boundary_x="periodic",
    winding_sector=w00,
)

sector_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
)
sector_basis = sector_model.build_basis(solver="dfs")
exact_w00 = evaluate_local_witness_on_diagonal_ensemble(
    witness,
    basis_configs=sector_basis.states,
)

assert sector_basis.n_states == 132
assert np.isclose(transfer_w00.expectation, exact_w00.expectation)
pd.DataFrame(
    [
        {
            "method": "explicit w00 basis",
            "dimension": sector_basis.n_states,
            "<Q_R>": exact_w00.expectation,
        },
        {
            "method": "winding-resolved transfer",
            "dimension": transfer_w00.partition_count,
            "<Q_R>": transfer_w00.expectation,
        },
    ]
)

## 6. Fourier projection of the winding charge

The small-width reference backend carries every $w_y$ charge explicitly.  The Fourier backend instead evaluates a charge-twisted transfer matrix at roots of unity and extracts the desired coefficient with a discrete Fourier transform.  With the default `fourier_points=Lx+1`, the projection is exact.


In [ ]:
fourier_w00 = transfer.evaluate_witness(
    placement,
    length=reference_model.lx,
    boundary_x="periodic",
    winding_sector=w00,
    winding_projection="fourier",
)
fourier_seam = transfer.evaluate_witness(
    placement,
    length=reference_model.lx,
    boundary_x="periodic",
    insertion_x=3,
    winding_sector=w00,
    winding_projection="fourier",
)

assert np.isclose(fourier_w00.expectation, transfer_w00.expectation)
assert np.isclose(fourier_seam.expectation, transfer_w00.expectation)
pd.DataFrame(
    [
        {
            "backend": "charge-resolved dynamic programming",
            "dimension": transfer_w00.partition_count,
            "weighted count": transfer_w00.weighted_count,
            "<Q_R>": transfer_w00.expectation,
            "exact": True,
        },
        {
            "backend": "Fourier projection",
            "dimension": fourier_w00.partition_count,
            "weighted count": fourier_w00.weighted_count,
            "<Q_R>": fourier_w00.expectation,
            "exact": fourier_w00.metadata["exact_fourier_projection"],
        },
    ]
)

## 7. Increase the x length at fixed circumference

The open-x sequence places the witness in the center.  The transfer vectors are normalized after every column, so very long strips do not overflow even though the number of coverings grows exponentially.

Use even and odd length sequences separately when parity oscillations are visible.


In [ ]:
lengths = tuple(range(4, 66, 2))
strip_scaling = transfer.scan_witness(
    placement,
    lengths=lengths,
    boundary_x="open",
)

strip_table = pd.DataFrame(
    {
        "Lx": strip_scaling.lengths,
        "<Q_R>": strip_scaling.expectations,
        "log number of coverings": [
            evaluation.log_partition_count for evaluation in strip_scaling.evaluations
        ],
    }
)
strip_table.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.plot(strip_table["Lx"], strip_table["<Q_R>"], marker="o")
ax.set_xlabel("strip length $L_x$")
ax.set_ylabel(r"$\langle Q_R\rangle_{\beta=0}$")
ax.set_title(f"Square-QDM cylinder, circumference $L_y={reference_model.ly}$")
ax.grid(alpha=0.25)
plt.show()

strip_scaling.tail_estimate(tail_points=5)

## 8. Repeat at several circumferences

The controlled route toward the two-dimensional limit is nested:

1. extrapolate $L_x\to\infty$ at each fixed $L_y$;
2. then inspect the dependence on $L_y$.

The cell below uses a centered open strip and the same local plaquette witness at each circumference.  Increasing `environment_columns` improves the first extrapolation without changing the boundary-state dimension $2^{L_y}$.


In [ ]:
def plaquette_flip_placement(circumference: int):
    model = SquareQDMModel(
        lx=4,
        ly=circumference,
        boundary_condition="periodic",
    )
    pid = model.lattice.plaquette_id_from_cell(0, 0)
    indices = tuple(
        model.layout.link_variable_index(int(link_id))
        for link_id in model.lattice.plaquette_links(pid)
    )
    support = ReducedIZPatternSupport(
        pattern_key=(
            (
                (1, 0, 1, 0),
                (0, 1, 0, 1),
                (1.0, 0.0),
            ),
        ),
        variable_indices=indices,
        source_zero_indices=(),
        mechanism_labels=(),
    )
    local_template = local_witness_template_from_pattern_support(support)
    local_witness = local_template.instantiate(indices)
    return SquareQDMWitnessPlacement.from_local_witness(model, local_witness)


environment_columns = 32
width_rows = []
for circumference in (2, 4, 6, 8):
    local_placement = plaquette_flip_placement(circumference)
    local_transfer = SquareQDMStripTransferMatrix(circumference)
    total_length = 2 * environment_columns + local_placement.window_width
    evaluation = local_transfer.evaluate_witness(
        local_placement,
        length=total_length,
        boundary_x="open",
        insertion_x=environment_columns,
    )
    width_rows.append(
        {
            "Ly": circumference,
            "boundary states": local_transfer.n_boundary_states,
            "Lx": total_length,
            "<Q_R>": evaluation.expectation,
        }
    )

width_table = pd.DataFrame(width_rows)
width_table

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.plot(width_table["Ly"], width_table["<Q_R>"], marker="o")
ax.set_xlabel("cylinder circumference $L_y$")
ax.set_ylabel(r"centered-strip $\langle Q_R\rangle_{\beta=0}$")
ax.grid(alpha=0.25)
plt.show()

## 9. Lift the old winding-resolution width ceiling

For `Ly=10` there are $2^{10}=1024$ transfer boundary states, beyond the previous 512-state dynamic-programming limit.  `winding_projection="auto"` selects the dense Fourier block method.  The central $w_x=0$ block has only 252 states.


In [ ]:
wide_placement = plaquette_flip_placement(10)
wide_transfer = SquareQDMStripTransferMatrix(circumference=10)
wide_w00 = wide_transfer.evaluate_witness(
    wide_placement,
    length=4,
    boundary_x="periodic",
    winding_sector=(0, 0),
)
{
    "boundary states": wide_transfer.n_boundary_states,
    "w00 block states": wide_w00.metadata["n_x_boundary_states"],
    "backend": wide_w00.metadata["contraction"],
    "exact Fourier projection": wide_w00.metadata["exact_fourier_projection"],
    "dimension": wide_w00.partition_count,
    "<Q_R>": wide_w00.expectation,
}

## 10. Replace the demonstration witness by a cage-derived witness

After classifying a cage state, the same workflow starts with:

```python
from qlinks.caging import local_witnesses_from_classification_report

witnesses = local_witnesses_from_classification_report(classification_report)
witness = witnesses[0]
placement = SquareQDMWitnessPlacement.from_local_witness(model, witness)
transfer = SquareQDMStripTransferMatrix(circumference=model.ly)
scaling = transfer.scan_witness(
    placement,
    lengths=(8, 12, 16, 24, 32, 48),
    boundary_x="open",
)
```

For an ETH claim, additionally verify on each finite cage state that
$\langle Q_R\rangle=\operatorname{Var}(Q_R)=0$, then pass the cage sector through `winding_sector=(w_x, w_y)` in the periodic transfer calculation.


## 11. Evaluate a common witness family directly

`common_local_witness_families(...)` returns the same local template with its embeddings at several sizes.  The helper below converts each embedding to strip coordinates and evaluates it in the requested sector.  Here we build a minimal two-size family explicitly so the notebook remains self-contained.


In [ ]:
model_6x4 = SquareQDMModel(lx=6, ly=4, boundary_condition="periodic")
pid_6x4 = model_6x4.lattice.plaquette_id_from_cell(0, 0)
indices_6x4 = tuple(
    model_6x4.layout.link_variable_index(int(link_id))
    for link_id in model_6x4.lattice.plaquette_links(pid_6x4)
)
witness_6x4 = template.instantiate(indices_6x4)
family = LocalWitnessFamily(
    template=template,
    embeddings=(
        LocalWitnessEmbeddingRecord("4x4", (witness,)),
        LocalWitnessEmbeddingRecord("6x4", (witness_6x4,)),
    ),
)
family_strip = evaluate_square_qdm_witness_family_on_strips(
    family,
    models={"4x4": reference_model, "6x4": model_6x4},
    lengths={"4x4": (4,), "6x4": (6,)},
    boundary_x="periodic",
    winding_sector=(0, 0),
)
pd.DataFrame(
    [
        {
            "system": record.system_label,
            "Lx": record.scaling_report.lengths[0],
            "<Q_R>_w00": record.scaling_report.expectations[0],
        }
        for record in family_strip.records
    ]
)

## 12. Factorized cage-family certification

The complementary eigenstate-side API avoids materializing the Cartesian product support of separated cage blocks:

```python
from qlinks.caging import (
    certify_qdm_factorized_product_state,
    find_factorized_qdm_block_paddings,
)

paddings = find_factorized_qdm_block_paddings(
    model,
    selected_blocks,
    config=padding_config,
)
certificate = certify_qdm_factorized_product_state(
    model,
    selected_blocks,
    paddings[0],
    config=padding_config,
)
certificate.to_summary_dict()
```

This is the eigenstate certificate; the strip transfer matrix supplies the independent thermal expectation of the local witness.


## 13. Extract and normalize the actual stripe-cage witness

The preceding sections used a directed plaquette flip as a transparent benchmark. We now run the local stripe search and select the rotated $(0,4)$ cage whose certified unit cell repeats along $x$. This aligns the exact cage sequence, the winding-sector transfer geometry, and the **actual reduced-IZ row operator** in the same $(4n)\times4$ systems. `normalization="operator_norm"` fixes

\[
\lVert Q_R\rVert=\lVert L_R\rVert^2=1,
\]

so the thermal number has an invariant scale.


In [ ]:
stripe_model = SquareQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_x=0,
    winding_y=0,
    winding_convention="electric",
    coup_kin=1.0,
    coup_pot=1.0,
)
stripe_config = RobustQDMLocalCageSearchConfig(
    local_config=LocalQDMCageSearchConfig(
        halo_layers=0,
        boundary_mode="relaxed",
        prune_inactive_local_basis_states=True,
        tolerance=1.0e-10,
        degenerate_basis_strategy="ipr",
    ),
    region_strategies=("stripe",),
    stripe_widths=(1,),
    stripe_directions=(0, 1),
    max_regions_per_strategy=None,
    block_signatures=((0, 2),),
    max_records_per_region=2,
    min_blocks=2,
    max_blocks=None,
    max_product_support_size=2048,
    max_paddings_per_stage=100,
    max_paddings_per_packing=10,
    include_sectors=True,
    padding_stages=("static",),
    tolerance=1.0e-9,
    store_full_states=False,
)
stripe_certified, stripe_context = robust_qdm_local_cage_search(
    stripe_model,
    config=stripe_config,
    return_context=True,
)
stripe_certified.counts_by_signature

In [ ]:
repeatable_report_index = 4
stripe_record = stripe_certified.records[repeatable_report_index]
stripe_classification = classify_cage_state(
    stripe_record.cage_state,
    kinetic_matrix=stripe_certified.kinetic_matrix,
    basis_configs=stripe_certified.basis.states,
    hilbert_size=stripe_certified.hilbert_size,
    config=CageClassificationConfig(sector_policy="infer_support_component"),
)
actual_strip = evaluate_square_qdm_classification_witnesses_on_strips(
    stripe_classification,
    model=stripe_model,
    lengths=(4, 8, 12, 16, 20),
    winding_sector=(0, 0),
    normalization="operator_norm",
    winding_projection="fourier",
)
pd.DataFrame(
    [
        {
            "witness": record.witness_index,
            "||Q_R||": record.witness.q_operator_norm,
            "last Lx": record.scaling_report.lengths[-1],
            "last <Q_R>_w00": record.scaling_report.expectations[-1],
            "tail minimum": record.scaling_report.tail_estimate()["minimum"],
        }
        for record in actual_strip.records
    ]
)


In [ ]:
actual_witness = actual_strip.records[0].witness
actual_placement = actual_strip.records[0].placement
width_rows = []
for ly in (4, 6, 8):
    wider = actual_placement.with_circumference(ly)
    result = SquareQDMStripTransferMatrix(ly).evaluate_witness(
        wider,
        length=32,
        boundary_x="open",
    )
    width_rows.append({"Ly": ly, "Lx": 32, "<Q_R>": result.expectation})
actual_width_table = pd.DataFrame(width_rows)
actual_width_table

## 14. Prove an infinite cage-state sequence

The selected stripe cage consists of two coherent $(0,2)$ factors and static spacer links. The sequence certificate does not infer exactness from a small global residual. It checks the stronger local decomposition:

1. every factor has a support-independent dimer count and winding contribution;
2. every plaquette touching two factors is frozen for every local support combination;
3. plaquettes touching one factor close on that factor with a fixed kinetic eigenvalue and constant potential;
4. three translated cells expose both neighbors for the range-one plaquette Hamiltonian.

The one- and two-cell rings are checked separately. Together these checks prove exact states on every $(4n)\times4$ torus for every positive integer $n$, all in the electric $w_{00}$ sector. This is a rigorous fixed-width thermodynamic sequence; both linear dimensions do not yet diverge.


In [ ]:
stripe_unit_cell = SquareQDMPeriodicProductUnitCell.from_padding(
    stripe_model,
    stripe_context.blocks,
    stripe_certified.reports[repeatable_report_index].padding,
    repeat_axis="x",
)
stripe_sequence = certify_square_qdm_periodic_product_sequence(stripe_unit_cell)
{
    "certified": stripe_sequence.is_certified,
    "minimum n": stripe_sequence.minimum_proven_repeats,
    "support per unit cell": stripe_sequence.support_size_per_unit_cell,
    "E_unit": stripe_sequence.unit_cell_energy,
    "E/N": stripe_sequence.energy_density,
    "unit-cell winding": stripe_sequence.unit_cell_winding_sector,
    "winding at n=6": stripe_sequence.winding_sector_for_repeats(6),
    "formal support at n=6": stripe_sequence.formal_support_size(6),
}


In [ ]:
sequence_witness = certify_local_witness_on_square_qdm_periodic_sequence(
    stripe_sequence,
    local_witnesses_from_classification_report(stripe_classification)[0],
    normalization="operator_norm",
)
sequence_witness.to_summary_dict()

## 15. Match the exact cage energy density to the beta-zero shell

For the model with a nonzero potential coupling, the fixed-width beta-zero energy density is not exactly $1/4$; that version requires a finite-temperature matcher at some $\beta_*$ and should not be called an infinite-temperature ETH result yet.

There is, however, an exact route already available. Reuse the same geometric cage cell for the **pure kinetic QDM**, $V=0$. The local sequence certificate then gives

$$
E_n=0,\qquad e_{\rm cage}=0.
$$

Since the kinetic Hamiltonian is off-diagonal in the dimer basis, its trace vanishes in every finite winding sector, so $e_{\beta=0}=0$ exactly at every size. The cage sequence and the beta-zero shell therefore match without extrapolation.


In [ ]:
kinetic_unit_cell = stripe_unit_cell.with_couplings(coup_pot=0.0)
kinetic_sequence = certify_square_qdm_periodic_product_sequence(kinetic_unit_cell)
beta_zero_energy = scan_square_qdm_beta_zero_energy_density(
    ((4, 4), (8, 4), (12, 4), (16, 4), (20, 4)),
    potential_coupling=0.0,
    winding_sector=(0, 0),
    winding_projection="fourier",
)
energy_table = pd.DataFrame(
    [
        {
            "Lx": item.length,
            "Ly": item.circumference,
            "e_beta0": item.energy_density,
            "e_cage": kinetic_sequence.energy_density,
            "difference": item.energy_density - kinetic_sequence.energy_density,
        }
        for item in beta_zero_energy.evaluations
    ]
)
energy_table


In [ ]:
exact_match = kinetic_sequence.match_energy_density(
    beta_zero_energy.evaluations[-1].energy_density,
    tolerance=1.0e-12,
    metadata={"geometry": "(4n)x4", "hamiltonian": "pure kinetic square QDM"},
)
{
    **exact_match.to_summary_dict(),
    "sequence winding at n=5": kinetic_sequence.winding_sector_for_repeats(5),
    "actual witness remains dark": certify_local_witness_on_square_qdm_periodic_sequence(
        kinetic_sequence,
        local_witnesses_from_classification_report(stripe_classification)[0],
        normalization="operator_norm",
    ).is_infinite_sequence_witness,
}


## 16. Diagnose true two-dimensional repetition

The one-axis stripe theorem does **not** imply a two-dimensional thermodynamic family.  A bi-periodic product tile must remain exact when copies are glued independently in both directions.  The diagnostic below builds a $3\times3$ array of the known stripe tile and separates four local environments:

- tile interiors;
- plaquettes and sites crossing an $x$ tile seam;
- plaquettes and sites crossing a $y$ tile seam;
- four-tile corners.

For a range-one square-QDM Hamiltonian, closure of these environments, together with the separately checked short tori, is sufficient for arbitrary independent repeat counts $(n_x,n_y)$.

In [ ]:
stripe_biperiodic_tile = SquareQDMBiperiodicProductTile.from_padding(
    stripe_model,
    stripe_context.blocks,
    stripe_certified.reports[repeatable_report_index].padding,
)
stripe_biperiodic = diagnose_square_qdm_biperiodic_repeatability(
    stripe_biperiodic_tile,
    verification_repeats=3,
    check_smaller_repeats=False,
)
failed = stripe_biperiodic.failed_checks[0]
{
    "certified true 2D family": stripe_biperiodic.is_certified,
    "failure reason": failed.failure_reason,
    "interior constraint residual": (
        failed.seam_diagnostics.max_site_constraint_residuals["internal"]
    ),
    "x-seam constraint residual": (
        failed.seam_diagnostics.max_site_constraint_residuals["x_seam"]
    ),
    "y-seam constraint residual": (
        failed.seam_diagnostics.max_site_constraint_residuals["y_seam"]
    ),
    "corner constraint residual": (
        failed.seam_diagnostics.max_site_constraint_residuals["corner"]
    ),
}

In [ ]:
pd.DataFrame(
    [
        {
            "environment": name,
            "plaquettes": failed.seam_diagnostics.plaquette_counts[name],
            "inert pattern checks": failed.seam_diagnostics.inert_pattern_checks[name],
            "flippable inert patterns": (
                failed.seam_diagnostics.flippable_inert_patterns[name]
            ),
            "max dimer residual": (
                failed.seam_diagnostics.max_site_constraint_residuals[name]
            ),
        }
        for name in ("internal", "x_seam", "y_seam", "corner")
    ]
)

The expected result is that the tile interior still closes, while the transverse seam develops support-dependent dimer charge and active plaquettes.  This localizes the obstruction: simply stacking the existing stripe is impossible as an independent product tile, but it does not rule out a different contractible tile.

## 17. Search finite periodic tiles directly

The search below reuses the local $(0,2)$ cage-block pool, solves only the finite $4\times4$ periodic exterior, and then applies the $3\times3$ bi-periodic diagnostic.  The repeated-system support is never materialized.  Failed candidates are retained and grouped by obstruction, which is useful for deciding whether to enlarge the tile, change the local blocks, or move to boundary-correlated tensor gluing.

In [ ]:
biperiodic_search = search_square_qdm_biperiodic_product_tiles(
    stripe_model,
    stripe_context.blocks,
    config=SquareQDMBiperiodicTileSearchConfig(
        min_blocks=2,
        max_blocks=2,
        max_padding_attempts=16,
        max_paddings_per_packing=1,
        max_results=16,
        max_tile_support_size=64,
        verification_repeats=3,
        check_smaller_repeats=False,
        require_kinetic_separation=False,
    ),
)
biperiodic_search.to_summary_dict()

In [ ]:
pd.DataFrame(
    [record.to_summary_dict() for record in biperiodic_search.records]
).sort_values(["is_certified", "score"], ascending=[False, False])

At the current milestone this is a **search and no-go diagnostic**, not yet a nontrivial two-dimensional cage theorem.  The known stripe-derived $4\times4$ product tiles fail at seams or corners.  The next numerical expansion should scan larger periods and finite contractible local-cage proposals, while keeping this same local $3\times3$ acceptance test.